In [ ]:

!pip install -q ultralytics sahi
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, math, random
import cv2, numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/aerial_task'
VID  = f'{ROOT}/videos'
CLIPS = ['train_A', 'train_B', 'train_C', 'train_D', 'eval']
print(os.listdir(VID))


In [ ]:
SAMPLE_FPS = {'train_A': 2, 'train_B': 2, 'train_C': 2, 'train_D': 2, 'eval': 5}
meta = {}
for clip in CLIPS:
    out = f'{ROOT}/frames/{clip}'
    os.makedirs(out, exist_ok=True)
    cap = cv2.VideoCapture(f'{VID}/{clip}.mp4')
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = max(1, round(fps / SAMPLE_FPS[clip]))
    i = saved = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            h, w = frame.shape[:2]
            cv2.imwrite(f'{out}/{clip}_{i:06d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        i += 1
    meta[clip] = dict(video_fps=fps, total_frames=i, saved=saved, step=step,
                      width=w, height=h, sample_fps=fps / step)
    print(clip, meta[clip])
json.dump(meta, open(f'{ROOT}/meta.json', 'w'), indent=2)

In [ ]:
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

YOLO('yolo11x.pt')   # просто щоб ваги завантажились
CONF = 0.25
det = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',   # якщо помилка на старій версії sahi: 'yolov8'
    model_path='yolo11x.pt', confidence_threshold=CONF, device='cuda:0')
VEH = {'car', 'truck', 'bus', 'motorcycle'}

rows = []
for clip in CLIPS:
    out = f'{ROOT}/labels_raw/{clip}'
    os.makedirs(out, exist_ok=True)
    for img in sorted(glob.glob(f'{ROOT}/frames/{clip}/*.jpg')):
        name = os.path.basename(img)[:-4]
        r = get_sliced_prediction(img, det, slice_height=640, slice_width=640,
                                  overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0)
        W, H = meta[clip]['width'], meta[clip]['height']
        lines = []
        for p in r.object_prediction_list:
            if p.category.name not in VEH:
                continue
            x1, y1, x2, y2 = p.bbox.to_xyxy()
            bw, bh = x2 - x1, y2 - y1
            lines.append(f'0 {(x1+bw/2)/W:.6f} {(y1+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}')
            rows.append(dict(clip=clip, frame=name, x1=x1, y1=y1, x2=x2, y2=y2,
                             conf=p.score.value, coco_cls=p.category.name))
        open(f'{out}/{name}.txt', 'w').write('\n'.join(lines))
    print(clip, 'done')

pred = pd.DataFrame(rows)
pred.to_csv(f'{ROOT}/prelabels.csv', index=False)
print(pred.groupby('clip').size())

In [ ]:
import zipfile, shutil

fr = pd.DataFrame([dict(clip=c, frame=os.path.basename(f)[:-4])
                   for c in CLIPS for f in sorted(glob.glob(f'{ROOT}/frames/{c}/*.jpg'))])
pred['w'] = pred.x2 - pred.x1
pred['h'] = pred.y2 - pred.y1
pred['long_px'] = pred[['w', 'h']].max(axis=1)
n   = pred.groupby(['clip', 'frame']).size().rename('n')
low = pred[pred.conf < 0.4].groupby(['clip', 'frame']).size().rename('n_low')
fr = fr.join(n, on=['clip', 'frame']).join(low, on=['clip', 'frame']).fillna(0)
fr['jump'] = fr.groupby('clip')['n'].diff().abs().fillna(0)
fr['suspicious'] = fr.jump + fr.n_low


In [ ]:
qa = pred.groupby('clip').agg(boxes=('conf', 'size'),
                              low_conf_share=('conf', lambda c: round((c < 0.4).mean(), 2)),
                              median_box_px=('long_px', 'median'))
qa['frames'] = fr.groupby('clip').size()
qa['frames_with_jumps'] = fr[fr.jump >= 3].groupby('clip').size()
print(qa.fillna(0))


In [ ]:
random.seed(0)
sample = {}
for c in CLIPS[:-1]:
    f = fr[fr['clip'] == c]
    top = f.nlargest(10, 'suspicious').frame.tolist()
    rest = [x for x in f.frame if x not in top]
    sample[c] = top + random.sample(rest, min(15, len(rest)))
e = fr[fr['clip'] == 'eval'].frame.tolist()
sample['eval'] = e[::max(1, len(e) // 60)][:60]
json.dump(sample, open(f'{ROOT}/sample.json', 'w'), indent=1)


In [ ]:
def make_cvat_zip(clips, name):
    os.makedirs(f'{ROOT}/cvat/{name}_images', exist_ok=True)
    lst = []
    with zipfile.ZipFile(f'{ROOT}/cvat/{name}_prelabels.zip', 'w') as z:
        z.writestr('obj.names', 'vehicle\n')
        z.writestr('obj.data', 'classes = 1\nnames = data/obj.names\ntrain = data/train.txt\n')
        for c in clips:
            for fname in sample[c]:
                z.write(f'{ROOT}/labels_raw/{c}/{fname}.txt', f'obj_train_data/{fname}.txt')
                shutil.copy(f'{ROOT}/frames/{c}/{fname}.jpg', f'{ROOT}/cvat/{name}_images/')
                lst.append(f'data/obj_train_data/{fname}.jpg')
        z.writestr('train.txt', '\n'.join(lst))

make_cvat_zip(CLIPS[:-1], 'train_sample')   # 100 кадрів
make_cvat_zip(['eval'], 'eval_sample')      # 60 кадрів


In [ ]:
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from IPython.display import Image as IPImage, display

meta = json.load(open(f'{ROOT}/meta.json'))
sample = json.load(open(f'{ROOT}/sample.json'))
VEH = {'car', 'truck', 'bus', 'motorcycle'}

det_low = AutoDetectionModel.from_pretrained(
    model_type='ultralytics', model_path='yolo11x.pt',
    confidence_threshold=0.05, device='cuda:0')

os.makedirs(f'{ROOT}/diag', exist_ok=True)
test = [sample[c][10] for c in CLIPS[:-1]]   # перший випадковий кадр з кожного train-кліпу (тільки train!)
allrows = []
for fname in test:
    c = fname.rsplit('_', 1)[0]
    img_path = f'{ROOT}/frames/{c}/{fname}.jpg'
    r = get_sliced_prediction(img_path, det_low, slice_height=640, slice_width=640,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0)
    img = cv2.imread(img_path)
    for p in r.object_prediction_list:
        x1, y1, x2, y2 = map(int, p.bbox.to_xyxy())
        name, conf = p.category.name, p.score.value
        allrows.append(dict(clip=c, frame=fname, cls=name, conf=conf))
        if name in VEH:
            color = (0, 255, 0) if conf >= 0.25 else (0, 255, 255)   # зелений: вже було; жовтий: дав би нижчий поріг
        else:
            color = (255, 0, 255)                                      # фіолетовий: інший клас COCO
            cv2.putText(img, name, (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    out = f'{ROOT}/diag/{fname}.jpg'
    cv2.imwrite(out, cv2.resize(img, (1600, int(1600 * img.shape[0] / img.shape[1]))))

d = pd.DataFrame(allrows)
d['bin'] = pd.cut(d['conf'], [0, 0.1, 0.25, 1.0])
ct = pd.crosstab(d['cls'], d['bin'])
ct['total'] = ct.sum(axis=1)
print(ct.sort_values('total', ascending=False))
for fname in test:
    print(fname)
    display(IPImage(f'{ROOT}/diag/{fname}.jpg', width=1000))

In [ ]:
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from IPython.display import Image as IPImage, display
import shutil, zipfile

meta = json.load(open(f'{ROOT}/meta.json'))
sample = json.load(open(f'{ROOT}/sample.json'))
VEH = {'car', 'truck', 'bus', 'motorcycle'}

def slice_px(W):
    # плитка ~1/6 ширини кадру: 1920 -> 320, 3840 -> 640 (машини збільшуються однаково для 1080p і 4K)
    return int(round(W / 6 / 32) * 32)

def run(img_path, det, s):
    r = get_sliced_prediction(img_path, det, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0)
    return [p for p in r.object_prediction_list if p.category.name in VEH]

det_old = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                             confidence_threshold=0.25, image_size=640, device='cuda:0')
det_new = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                             confidence_threshold=0.10, image_size=640, device='cuda:0')

os.makedirs(f'{ROOT}/diag', exist_ok=True)
test = [sample[c][10] for c in CLIPS[:-1]]   # тільки train-кадри
for fname in test:
    c = fname.rsplit('_', 1)[0]
    path = f'{ROOT}/frames/{c}/{fname}.jpg'
    s = slice_px(meta[c]['width'])
    old = run(path, det_old, 640)
    new = run(path, det_new, s)
    print(f'{fname}: було {len(old)} боксів  ->  стало {len(new)}  (плитка {s})')
    img = cv2.imread(path)
    for p in new:
        x1, y1, x2, y2 = map(int, p.bbox.to_xyxy()); cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for p in old:
        x1, y1, x2, y2 = map(int, p.bbox.to_xyxy()); cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 1)
    cv2.imwrite(f'{ROOT}/diag/{fname}.jpg', img)
    display(IPImage(f'{ROOT}/diag/{fname}.jpg', width=1000))


In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import shutil, zipfile

meta = json.load(open(f'{ROOT}/meta.json'))

# --- вибірка: бекап старої, нова = 3 підозрілі + 3 випадкові на train-кліп, eval без змін
if not os.path.exists(f'{ROOT}/sample_v1.json'):
    shutil.copy(f'{ROOT}/sample.json', f'{ROOT}/sample_v1.json')
s1 = json.load(open(f'{ROOT}/sample_v1.json'))
sample = {c: s1[c][:3] + s1[c][10:13] for c in CLIPS[:-1]}
sample['eval'] = s1['eval']
json.dump(sample, open(f'{ROOT}/sample.json', 'w'), indent=1)

# --- бекап першої предрозмітки
if not os.path.exists(f'{ROOT}/labels_raw_v1'):
    shutil.copytree(f'{ROOT}/labels_raw', f'{ROOT}/labels_raw_v1')

# --- фінальні налаштування розмітника (вибрані на train-кадрах)
VEH = {'car', 'truck', 'bus'}
CONF = 0.10
det = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                         confidence_threshold=CONF, image_size=640, device='cuda:0')
def slice_px(W):
    return int(round(W / 6 / 32) * 32)   # 1920 -> 320, 3840 -> 640

def prelabel(c, name):
    W, H = meta[c]['width'], meta[c]['height']
    s = slice_px(W)
    r = get_sliced_prediction(f'{ROOT}/frames/{c}/{name}.jpg', det, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              perform_standard_pred=False, verbose=0)
    lines, rows = [], []
    for p in r.object_prediction_list:
        if p.category.name not in VEH:
            continue
        x1, y1, x2, y2 = p.bbox.to_xyxy()
        bw, bh = x2 - x1, y2 - y1
        lines.append(f'0 {(x1+bw/2)/W:.6f} {(y1+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}')
        rows.append(dict(clip=c, frame=name, x1=x1, y1=y1, x2=x2, y2=y2,
                         conf=p.score.value, coco_cls=p.category.name))
    open(f'{ROOT}/labels_raw/{c}/{name}.txt', 'w').write('\n'.join(lines))
    return rows

allrows = []
for c in CLIPS:
    for name in sample[c]:
        allrows += prelabel(c, name)
    print(c, 'done')
pred_final = pd.DataFrame(allrows)
pred_final.to_csv(f'{ROOT}/prelabels_final.csv', index=False)
print(pred_final.groupby('clip').size())

# --- архіви для CVAT
def make_cvat_zip(clips, name):
    img_dir = f'{ROOT}/cvat/{name}_images'
    os.makedirs(img_dir, exist_ok=True)
    lst = []
    with zipfile.ZipFile(f'{ROOT}/cvat/{name}_prelabels.zip', 'w') as z:
        z.writestr('obj.names', 'vehicle\n')
        z.writestr('obj.data', 'classes = 1\nnames = data/obj.names\ntrain = data/train.txt\n')
        for c in clips:
            for fname in sample[c]:
                z.write(f'{ROOT}/labels_raw/{c}/{fname}.txt', f'obj_train_data/{fname}.txt')
                if not os.path.exists(f'{img_dir}/{fname}.jpg'):
                    shutil.copy(f'{ROOT}/frames/{c}/{fname}.jpg', img_dir)
                lst.append(f'data/obj_train_data/{fname}.jpg')
        z.writestr('train.txt', '\n'.join(lst))
    print(name, len(lst), 'кадрів')

make_cvat_zip(CLIPS[:-1], 'train_small')
make_cvat_zip(['eval'], 'eval_sample')

In [ ]:
import zipfile
from google.colab import files

sample = json.load(open(f'{ROOT}/sample.json'))
print('train_A_000540 у вибірці?', 'train_A_000540' in sample['train_A'])

with zipfile.ZipFile(f'{ROOT}/cvat/train_small_prelabels.zip') as z:
    lbls = sorted(os.path.basename(n) for n in z.namelist() if n.startswith('obj_train_data/'))
print('файлів розмітки:', len(lbls))

for name in ['train_small', 'eval_sample']:
    clips = CLIPS[:-1] if name == 'train_small' else ['eval']
    out = f'/content/{name}_images.zip'
    with zipfile.ZipFile(out, 'w') as z:
        for c in clips:
            for fname in sample[c]:
                z.write(f'{ROOT}/frames/{c}/{fname}.jpg', f'{fname}.jpg')   # картинки в корені архіву
    print(name, len(zipfile.ZipFile(out).namelist()), 'картинок')
    files.download(out)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, math, random, zipfile, shutil
import cv2, numpy as np, pandas as pd

ROOT  = '/content/drive/MyDrive/aerial_task'
CLIPS = ['train_A', 'train_B', 'train_C', 'train_D', 'eval']
meta   = json.load(open(f'{ROOT}/meta.json'))
sample = json.load(open(f'{ROOT}/sample.json'))

# параметри, зафіксовані до метрик (правило 1)
CAR_LEN_M, HFOV_DEG = 4.5, 80
IOU_THR, PROC_FPS   = 0.5, 5

def load_boxes(path, W, H):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(path, ndmin=2)
    cx, cy, w, h = a[:, 1] * W, a[:, 2] * H, a[:, 3] * W, a[:, 4] * H
    return np.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], 1)

def iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(a, b, thr):
    M = iou(a, b).copy(); pairs = []
    while M.size and M.max() >= thr:
        i, j = np.unravel_index(M.argmax(), M.shape)
        pairs.append((i, j, M[i, j])); M[i, :] = -1; M[:, j] = -1
    return pairs

def focal_px(W):
    return (W / 2) / math.tan(math.radians(HFOV_DEG / 2))

def distance_m(box, W):
    return focal_px(W) * CAR_LEN_M / max(box[2] - box[0], box[3] - box[1])

def band(d):
    return '0-200' if d < 200 else ('200-400' if d < 400 else '>400')

print('OK:', {c: len(v) for c, v in sample.items()})

In [ ]:
os.makedirs(f'{ROOT}/labels_fixed', exist_ok=True)
for zname in ['train_small_fixed.zip', 'eval_sample_fixed.zip']:
    p = f'{ROOT}/cvat/{zname}'
    assert os.path.exists(p), f'Немає файлу {p}: перевір назву на Drive'
    with zipfile.ZipFile(p) as z:
        txts = [m for m in z.namelist() if m.endswith('.txt') and '/' in m]
        for m in txts:
            open(f'{ROOT}/labels_fixed/' + os.path.basename(m), 'wb').write(z.read(m))
    print(zname, '->', len(txts), 'файлів розмітки')

edits = []
for c, frames_ in sample.items():
    W, H = meta[c]['width'], meta[c]['height']
    for f in frames_:
        raw = load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H)
        fix = load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H)
        pairs = match(raw, fix, 0.3)
        edits.append(dict(clip=c, frame=f, raw=len(raw), final=len(fix),
                          unchanged=sum(p[2] >= 0.9 for p in pairs),
                          fixed=sum(p[2] < 0.9 for p in pairs),
                          removed=len(raw) - len(pairs), added=len(fix) - len(pairs)))
edits = pd.DataFrame(edits)
edits.to_csv(f'{ROOT}/label_edits_per_frame.csv', index=False)

summary = edits.groupby('clip')[['raw', 'final', 'unchanged', 'fixed', 'removed', 'added']].sum()
summary.loc['TOTAL'] = summary.sum()
summary['raw_precision'] = (1 - summary['removed'] / summary['raw']).round(2)
summary['raw_miss_rate'] = (summary['added'] / summary['final']).round(2)
print(summary)
summary.to_csv(f'{ROOT}/label_edits.csv')

In [ ]:
import io
frame_names = {f for frames_ in sample.values() for f in frames_}

def extract_labels(zbytes, depth=0):
    found = 0
    with zipfile.ZipFile(io.BytesIO(zbytes)) as z:
        names = z.namelist()
        if depth == 0:
            print('   вміст (перші 12):', names[:12], '... всього', len(names))
        for m in names:
            base = os.path.basename(m)
            if m.endswith('.zip'):                                   # архів в архіві
                found += extract_labels(z.read(m), depth + 1)
            elif base.endswith('.txt') and base[:-4] in frame_names:
                open(f'{ROOT}/labels_fixed/{base}', 'wb').write(z.read(m))
                found += 1
    return found

shutil.rmtree(f'{ROOT}/labels_fixed', ignore_errors=True)
os.makedirs(f'{ROOT}/labels_fixed', exist_ok=True)
for zname in ['train_small_fixed.zip', 'eval_sample_fixed.zip']:
    p = f'{ROOT}/cvat/{zname}'
    print(zname)
    n = extract_labels(open(p, 'rb').read())
    print('   ->', n, 'файлів розмітки')

edits = []
for c, frames_ in sample.items():
    W, H = meta[c]['width'], meta[c]['height']
    for f in frames_:
        raw = load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H)
        fix = load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H)
        pairs = match(raw, fix, 0.3)
        edits.append(dict(clip=c, frame=f, raw=len(raw), final=len(fix),
                          unchanged=sum(p[2] >= 0.9 for p in pairs),
                          fixed=sum(p[2] < 0.9 for p in pairs),
                          removed=len(raw) - len(pairs), added=len(fix) - len(pairs)))
edits = pd.DataFrame(edits)
edits.to_csv(f'{ROOT}/label_edits_per_frame.csv', index=False)

summary = edits.groupby('clip')[['raw', 'final', 'unchanged', 'fixed', 'removed', 'added']].sum()
summary.loc['TOTAL'] = summary.sum()
summary['raw_precision'] = (1 - summary['removed'] / summary['raw']).round(2)
summary['raw_miss_rate'] = (summary['added'] / summary['final']).round(2)
pd.set_option('display.width', 200)
print(summary)
summary.to_csv(f'{ROOT}/label_edits.csv')


In [ ]:
import xml.etree.ElementTree as ET

frame_names = {f for frames_ in sample.values() for f in frames_}
shutil.rmtree(f'{ROOT}/labels_fixed', ignore_errors=True)
os.makedirs(f'{ROOT}/labels_fixed', exist_ok=True)

for zname in ['train_small_fixed.zip', 'eval_sample_fixed.zip']:
    with zipfile.ZipFile(f'{ROOT}/cvat/{zname}') as z:
        root = ET.fromstring(z.read('annotations.xml'))
    n_img = n_box = 0
    for im in root.iter('image'):
        name = os.path.splitext(os.path.basename(im.get('name')))[0]
        if name not in frame_names:
            continue
        W, H = float(im.get('width')), float(im.get('height'))
        lines = []
        for b in im.findall('box'):
            if b.get('label') != 'vehicle':
                continue
            x1, y1, x2, y2 = (float(b.get(k)) for k in ('xtl', 'ytl', 'xbr', 'ybr'))
            bw, bh = x2 - x1, y2 - y1
            lines.append(f'0 {(x1+bw/2)/W:.6f} {(y1+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}')
        open(f'{ROOT}/labels_fixed/{name}.txt', 'w').write('\n'.join(lines))
        n_img += 1; n_box += len(lines)
    print(f'{zname}: {n_img} кадрів, {n_box} боксів')

edits = []
for c, frames_ in sample.items():
    W, H = meta[c]['width'], meta[c]['height']
    for f in frames_:
        raw = load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H)
        fix = load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H)
        pairs = match(raw, fix, 0.3)
        edits.append(dict(clip=c, frame=f, raw=len(raw), final=len(fix),
                          unchanged=sum(p[2] >= 0.9 for p in pairs),
                          fixed=sum(p[2] < 0.9 for p in pairs),
                          removed=len(raw) - len(pairs), added=len(fix) - len(pairs)))
edits = pd.DataFrame(edits)
edits.to_csv(f'{ROOT}/label_edits_per_frame.csv', index=False)

summary = edits.groupby('clip')[['raw', 'final', 'unchanged', 'fixed', 'removed', 'added']].sum()
summary.loc['TOTAL'] = summary.sum()
summary['raw_precision'] = (1 - summary['removed'] / summary['raw']).round(2)
summary['raw_miss_rate'] = (summary['added'] / summary['final']).round(2)
pd.set_option('display.width', 200)
print(summary)
summary.to_csv(f'{ROOT}/label_edits.csv')

In [ ]:
rows = []
for c, frames_ in sample.items():
    W, H = meta[c]['width'], meta[c]['height']
    for f in frames_:
        raw = load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H)
        fix = load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H)
        pairs = match(raw, fix, 0.3)
        mr = {i for i, _, _ in pairs}; mf = {j for _, j, _ in pairs}
        r_un = [i for i in range(len(raw)) if i not in mr]
        f_un = [j for j in range(len(fix)) if j not in mf]
        M = iou(raw[r_un], fix[f_un]) if r_un and f_un else np.zeros((len(r_un), 0))
        for k, i in enumerate(r_un):
            best = M[k].max() if M.shape[1] else 0
            rows.append(dict(clip=c, frame=f, long_px=max(raw[i][2]-raw[i][0], raw[i][3]-raw[i][1]), best_iou=best))
rem = pd.DataFrame(rows)
rem['kind'] = pd.cut(rem['best_iou'], [-0.01, 0.0, 0.1, 0.3], labels=['нічого поруч', 'IoU 0-0.1', 'IoU 0.1-0.3 (перемальований?)'])
print(pd.crosstab(rem['clip'], rem['kind']))
print(rem.groupby('clip')['long_px'].median().round(0).rename('медіанний розмір видаленого бокса, px'))

# 3 eval-кадри: червоне = модель, зелене = після виправлення
from IPython.display import Image as IPImage, display
W, H = meta['eval']['width'], meta['eval']['height']
for f in sample['eval'][::20][:3]:
    img = cv2.imread(f'{ROOT}/frames/eval/{f}.jpg')
    for b in load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H):
        cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 0), 3)
    for b in load_boxes(f'{ROOT}/labels_raw/eval/{f}.txt', W, H):
        cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    cv2.imwrite(f'/content/check_{f}.jpg', img)
    print(f); display(IPImage(f'/content/check_{f}.jpg', width=1100))

In [ ]:
def focal_px(W, H):
    return (max(W, H) / 2) / math.tan(math.radians(HFOV_DEG / 2))   # 80° по довшій стороні кадру

def distance_m(box, W, H):
    return focal_px(W, H) * CAR_LEN_M / max(box[2] - box[0], box[3] - box[1])

print('eval meta:', meta['eval'])
W, H = meta['eval']['width'], meta['eval']['height']
fp = focal_px(W, H)
print(f'100 м = {fp*4.5/100:.0f} px, 200 м = {fp*4.5/200:.0f} px, 400 м = {fp*4.5/400:.0f} px')

gt_rows = []
for f in sample['eval']:
    for b in load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H):
        d = distance_m(b, W, H)
        gt_rows.append(dict(frame=f, long_px=max(b[2]-b[0], b[3]-b[1]), dist_m=d, band=band(d)))
gt = pd.DataFrame(gt_rows)
gt.to_csv(f'{ROOT}/eval_gt_distances.csv', index=False)
print(gt['band'].value_counts())
print(gt['dist_m'].describe().round(0))

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from google.colab import files

clip = 'eval2'
# --- 1. кадри, 5 fps (як eval)
out = f'{ROOT}/frames/{clip}'; os.makedirs(out, exist_ok=True)
cap = cv2.VideoCapture(f'{ROOT}/videos/{clip}.mp4')
fps = cap.get(cv2.CAP_PROP_FPS); step = max(1, round(fps / 5)); i = saved = 0
while True:
    ok, frame = cap.read()
    if not ok: break
    if i % step == 0:
        h, w = frame.shape[:2]
        cv2.imwrite(f'{out}/{clip}_{i:06d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95]); saved += 1
    i += 1
meta[clip] = dict(video_fps=fps, total_frames=i, saved=saved, step=step, width=w, height=h, sample_fps=fps / step)
json.dump(meta, open(f'{ROOT}/meta.json', 'w'), indent=2)
print(clip, meta[clip])

# --- 2. вибірка: 30 кадрів рівномірно
allf = sorted(os.path.basename(p)[:-4] for p in glob.glob(f'{out}/*.jpg'))
sample[clip] = allf[::max(1, len(allf) // 30)][:30]
json.dump(sample, open(f'{ROOT}/sample.json', 'w'), indent=1)

# --- 3. предрозмітка ТИМИ САМИМИ налаштуваннями, що й для eval (зафіксовані до метрик)
VEH = {'car', 'truck', 'bus'}
det = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                         confidence_threshold=0.10, image_size=640, device='cuda:0')
s = int(round(w / 6 / 32) * 32)
os.makedirs(f'{ROOT}/labels_raw/{clip}', exist_ok=True)
for name in sample[clip]:
    r = get_sliced_prediction(f'{out}/{name}.jpg', det, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              perform_standard_pred=False, verbose=0)
    lines = []
    for p in r.object_prediction_list:
        if p.category.name not in VEH: continue
        x1, y1, x2, y2 = p.bbox.to_xyxy(); bw, bh = x2 - x1, y2 - y1
        lines.append(f'0 {(x1+bw/2)/w:.6f} {(y1+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}')
    open(f'{ROOT}/labels_raw/{clip}/{name}.txt', 'w').write('\n'.join(lines))
print('prelabels done, плитка', s)

# --- 4. архіви для CVAT: картинки + предрозмітка
with zipfile.ZipFile(f'/content/{clip}_images.zip', 'w') as z:
    for name in sample[clip]:
        z.write(f'{out}/{name}.jpg', f'{name}.jpg')
with zipfile.ZipFile(f'/content/{clip}_prelabels.zip', 'w') as z:
    z.writestr('obj.names', 'vehicle\n')
    z.writestr('obj.data', 'classes = 1\nnames = data/obj.names\ntrain = data/train.txt\n')
    for name in sample[clip]:
        z.write(f'{ROOT}/labels_raw/{clip}/{name}.txt', f'obj_train_data/{name}.txt')
    z.writestr('train.txt', '\n'.join(f'data/obj_train_data/{n}.jpg' for n in sample[clip]))
files.download(f'/content/{clip}_images.zip')

In [ ]:
!pip install -q ultralytics sahi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, math, random, zipfile, shutil
import cv2, numpy as np, pandas as pd

ROOT  = '/content/drive/MyDrive/aerial_task'
CLIPS = ['train_A', 'train_B', 'train_C', 'train_D', 'eval']
meta   = json.load(open(f'{ROOT}/meta.json'))
sample = json.load(open(f'{ROOT}/sample.json'))

# параметри, зафіксовані до метрик (правило 1)
CAR_LEN_M, HFOV_DEG = 4.5, 80
IOU_THR, PROC_FPS   = 0.5, 5

def load_boxes(path, W, H):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(path, ndmin=2)
    cx, cy, w, h = a[:, 1] * W, a[:, 2] * H, a[:, 3] * W, a[:, 4] * H
    return np.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], 1)

def iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(a, b, thr):
    M = iou(a, b).copy(); pairs = []
    while M.size and M.max() >= thr:
        i, j = np.unravel_index(M.argmax(), M.shape)
        pairs.append((i, j, M[i, j])); M[i, :] = -1; M[:, j] = -1
    return pairs

def focal_px(W, H):
    return (max(W, H) / 2) / math.tan(math.radians(HFOV_DEG / 2))   # 80° по довшій стороні кадру

def distance_m(box, W, H):
    return focal_px(W, H) * CAR_LEN_M / max(box[2] - box[0], box[3] - box[1])

def band(d):
    return '0-200' if d < 200 else ('200-400' if d < 400 else '>400')

if 'eval2' in sample and 'eval2' not in CLIPS:
    CLIPS.append('eval2')
print('OK:', {c: len(v) for c, v in sample.items()})

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from google.colab import files

clip = 'eval2'
# --- 1. кадри, 5 fps (як eval)
out = f'{ROOT}/frames/{clip}'; os.makedirs(out, exist_ok=True)
cap = cv2.VideoCapture(f'{ROOT}/videos/{clip}.mp4')
fps = cap.get(cv2.CAP_PROP_FPS); step = max(1, round(fps / 5)); i = saved = 0
while True:
    ok, frame = cap.read()
    if not ok: break
    if i % step == 0:
        h, w = frame.shape[:2]
        cv2.imwrite(f'{out}/{clip}_{i:06d}.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95]); saved += 1
    i += 1
meta[clip] = dict(video_fps=fps, total_frames=i, saved=saved, step=step, width=w, height=h, sample_fps=fps / step)
json.dump(meta, open(f'{ROOT}/meta.json', 'w'), indent=2)
print(clip, meta[clip])

# --- 2. вибірка: 30 кадрів рівномірно
allf = sorted(os.path.basename(p)[:-4] for p in glob.glob(f'{out}/*.jpg'))
sample[clip] = allf[::max(1, len(allf) // 30)][:30]
json.dump(sample, open(f'{ROOT}/sample.json', 'w'), indent=1)

# --- 3. предрозмітка ТИМИ САМИМИ налаштуваннями, що й для eval (зафіксовані до метрик)
VEH = {'car', 'truck', 'bus'}
det = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                         confidence_threshold=0.10, image_size=640, device='cuda:0')
s = int(round(w / 6 / 32) * 32)
os.makedirs(f'{ROOT}/labels_raw/{clip}', exist_ok=True)
for name in sample[clip]:
    r = get_sliced_prediction(f'{out}/{name}.jpg', det, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              perform_standard_pred=False, verbose=0)
    lines = []
    for p in r.object_prediction_list:
        if p.category.name not in VEH: continue
        x1, y1, x2, y2 = p.bbox.to_xyxy(); bw, bh = x2 - x1, y2 - y1
        lines.append(f'0 {(x1+bw/2)/w:.6f} {(y1+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}')
    open(f'{ROOT}/labels_raw/{clip}/{name}.txt', 'w').write('\n'.join(lines))
print('prelabels done, плитка', s)

# --- 4. архіви для CVAT: картинки + предрозмітка
with zipfile.ZipFile(f'/content/{clip}_images.zip', 'w') as z:
    for name in sample[clip]:
        z.write(f'{out}/{name}.jpg', f'{name}.jpg')
with zipfile.ZipFile(f'/content/{clip}_prelabels.zip', 'w') as z:
    z.writestr('obj.names', 'vehicle\n')
    z.writestr('obj.data', 'classes = 1\nnames = data/obj.names\ntrain = data/train.txt\n')
    for name in sample[clip]:
        z.write(f'{ROOT}/labels_raw/{clip}/{name}.txt', f'obj_train_data/{name}.txt')
    z.writestr('train.txt', '\n'.join(f'data/obj_train_data/{n}.jpg' for n in sample[clip]))
files.download(f'/content/{clip}_images.zip')

In [ ]:
files.download('/content/eval2_prelabels.zip')

In [ ]:
!pip install -q ultralytics sahi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, math, random, zipfile, shutil
import cv2, numpy as np, pandas as pd

ROOT  = '/content/drive/MyDrive/aerial_task'
CLIPS = ['train_A', 'train_B', 'train_C', 'train_D', 'eval']
meta   = json.load(open(f'{ROOT}/meta.json'))
sample = json.load(open(f'{ROOT}/sample.json'))

# параметри, зафіксовані до метрик (правило 1)
CAR_LEN_M, HFOV_DEG = 4.5, 80
IOU_THR, PROC_FPS   = 0.5, 5

def load_boxes(path, W, H):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(path, ndmin=2)
    cx, cy, w, h = a[:, 1] * W, a[:, 2] * H, a[:, 3] * W, a[:, 4] * H
    return np.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], 1)

def iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(a, b, thr):
    M = iou(a, b).copy(); pairs = []
    while M.size and M.max() >= thr:
        i, j = np.unravel_index(M.argmax(), M.shape)
        pairs.append((i, j, M[i, j])); M[i, :] = -1; M[:, j] = -1
    return pairs

def focal_px(W, H):
    return (max(W, H) / 2) / math.tan(math.radians(HFOV_DEG / 2))   # 80° по довшій стороні кадру

def distance_m(box, W, H):
    return focal_px(W, H) * CAR_LEN_M / max(box[2] - box[0], box[3] - box[1])

def band(d):
    return '0-200' if d < 200 else ('200-400' if d < 400 else '>400')

if 'eval2' in sample and 'eval2' not in CLIPS:
    CLIPS.append('eval2')
print('OK:', {c: len(v) for c, v in sample.items()})

In [ ]:
import xml.etree.ElementTree as ET
if 'eval2' not in CLIPS: CLIPS.append('eval2')

frame_names = {f for frames_ in sample.values() for f in frames_}
shutil.rmtree(f'{ROOT}/labels_fixed', ignore_errors=True)
os.makedirs(f'{ROOT}/labels_fixed', exist_ok=True)

for zname in ['train_small_fixed.zip', 'eval_sample_fixed.zip', 'eval2_sample_fixed.zip']:
    with zipfile.ZipFile(f'{ROOT}/cvat/{zname}') as z:
        root = ET.fromstring(z.read('annotations.xml'))
    n_img = n_box = 0
    for im in root.iter('image'):
        name = os.path.splitext(os.path.basename(im.get('name')))[0]
        if name not in frame_names: continue
        W, H = float(im.get('width')), float(im.get('height'))
        lines = []
        for b in im.findall('box'):
            if b.get('label') != 'vehicle': continue
            x1, y1, x2, y2 = (float(b.get(k)) for k in ('xtl', 'ytl', 'xbr', 'ybr'))
            bw, bh = x2 - x1, y2 - y1
            lines.append(f'0 {(x1+bw/2)/W:.6f} {(y1+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}')
        open(f'{ROOT}/labels_fixed/{name}.txt', 'w').write('\n'.join(lines))
        n_img += 1; n_box += len(lines)
    print(f'{zname}: {n_img} кадрів, {n_box} боксів')

edits = []
for c, frames_ in sample.items():
    W, H = meta[c]['width'], meta[c]['height']
    for f in frames_:
        raw = load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H)
        fix = load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H)
        pairs = match(raw, fix, 0.3)
        mr = {i for i, _, _ in pairs}; mf = {j for _, j, _ in pairs}
        # "redrawn": видалений бокс моделі, поруч з яким намальовано новий (IoU 0.1-0.3), - фактично виправлення
        r_un = [i for i in range(len(raw)) if i not in mr]; f_un = [j for j in range(len(fix)) if j not in mf]
        redrawn = len(match(raw[r_un], fix[f_un], 0.1)) if r_un and f_un else 0
        edits.append(dict(clip=c, frame=f, raw=len(raw), final=len(fix),
                          unchanged=sum(p[2] >= 0.9 for p in pairs),
                          fixed=sum(p[2] < 0.9 for p in pairs),
                          redrawn=redrawn,
                          removed=len(raw) - len(pairs) - redrawn,
                          added=len(fix) - len(pairs) - redrawn))
edits = pd.DataFrame(edits)
edits.to_csv(f'{ROOT}/label_edits_per_frame.csv', index=False)
summary = edits.groupby('clip')[['raw', 'final', 'unchanged', 'fixed', 'redrawn', 'removed', 'added']].sum()
summary.loc['TOTAL'] = summary.sum()
summary['raw_precision'] = ((summary['unchanged'] + summary['fixed'] + summary['redrawn']) / summary['raw']).round(2)
summary['raw_miss_rate'] = (summary['added'] / summary['final']).round(2)
pd.set_option('display.width', 200)
print(summary)
summary.to_csv(f'{ROOT}/label_edits.csv')

In [ ]:
EVAL_CLIPS = ['eval', 'eval2']
gt_rows = []
for c in EVAL_CLIPS:
    W, H = meta[c]['width'], meta[c]['height']
    print(f'{c} {W}x{H}: 200 м = {focal_px(W,H)*4.5/200:.0f} px, 400 м = {focal_px(W,H)*4.5/400:.0f} px')
    for f in sample[c]:
        for b in load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H):
            d = distance_m(b, W, H)
            gt_rows.append(dict(clip=c, frame=f, long_px=max(b[2]-b[0], b[3]-b[1]), dist_m=d, band=band(d)))
gt = pd.DataFrame(gt_rows)
gt.to_csv(f'{ROOT}/eval_gt_distances.csv', index=False)
print(pd.crosstab(gt['clip'], gt['band'], margins=True))
print(gt.groupby('clip')['dist_m'].describe().round(0))


In [ ]:
def eval_metrics(iou_thr, raw_dir='labels_raw', clips=EVAL_CLIPS):
    stats = {b: dict(GT=0, TP=0, FP=0, FN=0) for b in ['0-200', '200-400', '>400']}
    N = 0
    for c in clips:
        W, H = meta[c]['width'], meta[c]['height']
        for f in sample[c]:
            N += 1
            g = load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H)
            p = load_boxes(f'{ROOT}/{raw_dir}/{c}/{f}.txt', W, H)
            pairs = match(p, g, iou_thr)
            mp = {i for i, _, _ in pairs}; mg = {j for _, j, _ in pairs}
            for j in range(len(g)):
                bj = band(distance_m(g[j], W, H))
                stats[bj]['GT'] += 1
                stats[bj]['TP' if j in mg else 'FN'] += 1           # TP/FN - за еталонним боксом
            for i in range(len(p)):
                if i not in mp:
                    stats[band(distance_m(p[i], W, H))]['FP'] += 1  # FP - за боксом моделі
    res = pd.DataFrame(stats).T
    res['detection_rate']  = (res['TP'] / (res['TP'] + res['FN'])).round(2)
    res['precision']       = (res['TP'] / (res['TP'] + res['FP'])).round(2)
    res['FA_per_min_5fps'] = (res['FP'] / N * PROC_FPS * 60).round(1)
    res['FA_task_formula'] = (res['FP'] * 60 / N).round(1)
    res['frames'] = N
    return res

res = eval_metrics(IOU_THR)
print(f'MAIN: IoU {IOU_THR}, eval + eval2'); print(res)
res.to_csv(f'{ROOT}/metrics.csv')

# розбивка по кліпах і м'якший IoU - побічні файли, не в головну таблицю
for c in EVAL_CLIPS:
    r = eval_metrics(IOU_THR, clips=[c]); r.to_csv(f'{ROOT}/metrics_{c}.csv')
    print(f'\n{c} only:'); print(r[['GT', 'TP', 'FP', 'FN', 'detection_rate', 'precision']])
eval_metrics(0.3).to_csv(f'{ROOT}/metrics_iou03.csv')

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
det = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                         confidence_threshold=0.10, image_size=640, device='cuda:0')
for c in EVAL_CLIPS:
    W, H = meta[c]['width'], meta[c]['height']; s = int(round(W / 6 / 32) * 32)
    os.makedirs(f'{ROOT}/labels_posthoc/{c}', exist_ok=True)
    for f in sample[c]:
        r = get_sliced_prediction(f'{ROOT}/frames/{c}/{f}.jpg', det, slice_height=s, slice_width=s,
                                  overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                                  perform_standard_pred=True, verbose=0)   # єдина зміна
        lines = []
        for p in r.object_prediction_list:
            if p.category.name not in {'car', 'truck', 'bus'}: continue
            x1, y1, x2, y2 = p.bbox.to_xyxy(); bw, bh = x2 - x1, y2 - y1
            lines.append(f'0 {(x1+bw/2)/W:.6f} {(y1+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}')
        open(f'{ROOT}/labels_posthoc/{c}/{f}.txt', 'w').write('\n'.join(lines))
post = eval_metrics(IOU_THR, raw_dir='labels_posthoc')
post.to_csv(f'{ROOT}/metrics_posthoc.csv')
print('POST-HOC (eval-informed, NOT a valid estimate):'); print(post)

In [ ]:
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
EVAL_CLIPS = ['eval', 'eval2']
ASSETS = f'{ROOT}/repo_assets'; os.makedirs(ASSETS, exist_ok=True)
COL = {'0-200': (0, 200, 0), '200-400': (0, 140, 255), '>400': (160, 160, 160)}

def resize_long(img, L):
    h, w = img.shape[:2]; k = L / max(h, w)
    return cv2.resize(img, (int(w * k), int(h * k)))

# --- 1. кадри: GT зеленим (товсто), модель червоним (тонко)
for c in EVAL_CLIPS:
    W, H = meta[c]['width'], meta[c]['height']
    for f in sample[c][::max(1, len(sample[c]) // 2)][:2]:
        img = cv2.imread(f'{ROOT}/frames/{c}/{f}.jpg')
        for b in load_boxes(f'{ROOT}/labels_fixed/{f}.txt', W, H):
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 0), 5)
        for b in load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H):
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 3)
        img = resize_long(img, 1600)
        cv2.putText(img, 'green = ground truth, red = model', (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        cv2.imwrite(f'{ASSETS}/{c}_{f}_gt_vs_pred.jpg', img)

# --- 2. відео з готових предиктів на кадрах вибірки, рамки кольором за дистанцією
for c in EVAL_CLIPS:
    W, H = meta[c]['width'], meta[c]['height']
    frames_ = sample[c]
    # реальна частота кадрів вибірки: 5 fps / крок між кадрами вибірки
    gap = int(frames_[1].rsplit('_', 1)[1]) - int(frames_[0].rsplit('_', 1)[1])
    out_fps = max(1.0, round(meta[c]['video_fps'] / gap, 1))
    first = resize_long(cv2.imread(f'{ROOT}/frames/{c}/{frames_[0]}.jpg'), 1280); oh, ow = first.shape[:2]
    tmp = f'/content/{c}_tmp.mp4'
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), out_fps, (ow, oh))
    for f in frames_:
        img = cv2.imread(f'{ROOT}/frames/{c}/{f}.jpg')
        for b in load_boxes(f'{ROOT}/labels_raw/{c}/{f}.txt', W, H):
            cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), COL[band(distance_m(b, W, H))], 5)
        img = resize_long(img, 1280)
        cv2.putText(img, 'model: green 0-200 m | orange 200-400 m | grey >400 m', (15, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        vw.write(img)
    vw.release()
    os.system(f'ffmpeg -y -loglevel error -i {tmp} -vcodec libx264 -pix_fmt yuv420p {ASSETS}/{c}_predictions.mp4')
    print(f'{c}: {len(frames_)} кадрів, {out_fps} fps, {os.path.getsize(f"{ASSETS}/{c}_predictions.mp4")/1e6:.1f} MB')

print(sorted(os.listdir(ASSETS)))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json

EVAL2_LINK = 'https://www.pexels.com/video/aerial-view-of-highway-in-vietnam-countryside-35695627/'

ROOT = '/content/drive/MyDrive/aerial_task'
meta = json.load(open(f'{ROOT}/meta.json'))
sample = json.load(open(f'{ROOT}/sample.json'))
links = {'train_A': 'pexels.com/video/8968356', 'train_B': 'pexels.com/video/5382494',
         'train_C': 'pexels.com/video/8457857', 'train_D': 'pexels.com/video/3405804',
         'eval': 'pexels.com/video/32179597', 'eval2': EVAL2_LINK}
roles = {'eval': 'eval', 'eval2': '**eval, added (rule 4)**'}

print('| Clip | Role | Source | Resolution | Length | Sampled at | Frames extracted | Frames reviewed |')
print('|---|---|---|---|---|---|---|---|')
for c in ['train_A', 'train_B', 'train_C', 'train_D', 'eval', 'eval2']:
    m = meta[c]
    name = '**eval2**' if c == 'eval2' else c
    orient = ' (portrait)' if m['height'] > m['width'] else ''
    print(f"| {name} | {roles.get(c, 'train')} | {links[c]} | {m['width']}×{m['height']}{orient} | "
          f"{m['total_frames']/m['video_fps']:.0f} s | {m['sample_fps']:.0f} fps | {m['saved']} | {len(sample[c])} |")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, shutil

ROOT = '/content/drive/MyDrive/aerial_task'
OUT = '/content/repo_extra'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(f'{OUT}/results')

for f in ['label_edits.csv', 'label_edits_per_frame.csv', 'eval_gt_distances.csv', 'metrics.csv',
          'metrics_eval.csv', 'metrics_eval2.csv', 'metrics_iou03.csv', 'meta.json', 'sample.json']:
    if os.path.exists(f'{ROOT}/{f}'):
        shutil.copy(f'{ROOT}/{f}', f'{OUT}/results/')
    else:
        print('немає файлу:', f)

for d in ['labels_raw', 'labels_fixed', 'labels_raw_v1']:
    if os.path.exists(f'{ROOT}/{d}'):
        shutil.copytree(f'{ROOT}/{d}', f'{OUT}/labels/{d}')
shutil.copytree(f'{ROOT}/repo_assets', f'{OUT}/assets')

shutil.make_archive('/content/repo_extra', 'zip', OUT)
for d in ['results', 'labels', 'assets']:
    n = sum(len(fs) for _, _, fs in os.walk(f'{OUT}/{d}'))
    print(d, n, 'файлів')
print('архів:', round(os.path.getsize('/content/repo_extra.zip') / 1e6, 1), 'MB')
files.download('/content/repo_extra.zip')